In [92]:
import pandas as pd 
import numpy as np

# Wages ([Statistics Canada - Wages from 2012 to 2024](https://open.canada.ca/data/en/dataset/adad580f-76b0-4502-bd05-20c125de9116))

In [93]:
# Load all wages datasets
df_2012 = pd.read_csv("../datasets/wages/2012 Wages.csv")
df_2013 = pd.read_csv("../datasets/wages/2013 Wages.csv")
df_2014 = pd.read_csv("../datasets/wages/2014 Wages.csv")
df_2015 = pd.read_csv("../datasets/wages/2015 Wages.csv")
df_2016 = pd.read_csv("../datasets/wages/2016 Wages.csv")

df_2019 = pd.read_csv("../datasets/wages/2019 Wages.csv")
df_2020 = pd.read_csv("../datasets/wages/2020 Wages.csv")
df_2021 = pd.read_csv("../datasets/wages/2021 Wages.csv")
df_2022 = pd.read_csv("../datasets/wages/2022 Wages.csv")
df_2023 = pd.read_csv("../datasets/wages/2023 Wages.csv", low_memory=False)
df_2024 = pd.read_csv("../datasets/wages/2024 Wages.csv")

# These 2 datasets have a different encoding
df_2017 = pd.read_csv("../datasets/wages/2017 Wages.csv", encoding='iso-8859-1') 
df_2018 = pd.read_csv("../datasets/wages/2018 Wages.csv", encoding='iso-8859-1')

## **Transformation**

#### **Combine all Wages datasets** 

We are dealing with 13 annual wage datasets (2012–2024).  
To enable efficient analysis and cross-year comparison, we must first standardize the column names and then combine them into a single unified DataFrame.

In [94]:
# Create list of dataframes with their corresponding years
dataframes = [
    (df_2012, 2012),
    (df_2013, 2013),
    (df_2014, 2014),
    (df_2015, 2015),
    (df_2016, 2016),
    (df_2017, 2017),
    (df_2018, 2018),
    (df_2019, 2019),
    (df_2020, 2020),
    (df_2021, 2021),
    (df_2022, 2022),
    (df_2023, 2023),
    (df_2024, 2024)
]

Check all present column names.

In [95]:
# Aggregate all unique column names from the 13 datasets
all_unique_columns = set()

# Iterate through the list of (dataframe, year) tuples
for df, year in dataframes:
    # Add all columns from the current DataFrame to the set
    all_unique_columns.update(df.columns.tolist())

# Convert the set back to a sorted list for clear display
sorted_unique_columns = sorted(list(all_unique_columns))

print("Total unique column names across all wage datasets:")
print(f"Count: {len(sorted_unique_columns)}")
print("-" * 50)
for col in sorted_unique_columns:
    print(col)

Total unique column names across all wage datasets:
Count: 49
--------------------------------------------------
Annual Wage Flag_Salaire annuel
Annual_Wage
Annual_Wage_Flag_Salaire_annuel
Average_Wage_Salaire_Moyen
Commentaire_Salaire
Data Source
Data_Source_E
Data_Source_F
ER Code_Code RE
ER Name_Nom RE
ER_Code
ER_Code_Code_RE
ER_Name
ER_Name_Nom_RE
Economic_Region
High Wage_Salaire maximal
High_Wage_Salaire_Maximal
Low Wage_Salaire Minium
Low_Wage_Salaire_Minium
Median wage_Salaire Median
Median_Wage_Salaire_Median
NOC Title
NOC_CNP
NOC_CNP_2006
NOC_TITLE_ENG
NOC_TITLE_FRA
NOC_Title
NOC_Title_E
NOC_Title_F
NOC_Title_eng
NOC_Title_fra
Nom_RE
Non_WageBen_pct
PROV
Prov
Quartile1_Wage_Salaire_Quartile1
Quartile3_Wage_Salaire_Quartile3
Reference Period_Periode de reference
Reference_Period
Revision Date_Date revision
Revision_Date_Date_revision
Source de donnees
Titre CNP
Titre_CNP
Wage_Comment
Wage_Comment_E
Wage_Comment_F
Wage_Comment_F1
prov


Finding **49 unique column names** across the 13 input files immediately highlights the major issue of inconsistent formatting.

In [96]:
# The column mappings (all lower-case)

# The column rename/standardization mapping
RENAME_MAP = {
    # NOC code
    'noc_cnp_2006': 'noc_code',
    'noc_cnp': 'noc_code',
    # NOC title
    'noc_title_e': 'noc_title',
    'noc_title_eng': 'noc_title',
    # Economic region info
    'er_name_nom_re': 'er_name',
    'economic_region': 'er_name',
    'er_code_code_re': 'er_code',
    # Wage info
    'low_wage_salaire_minium': 'low_wage',
    'high_wage_salaire_maximal': 'high_wage',
    'median_wage_salaire_median': 'median_wage',
    'annual_wage_flag_salaire_annuel': 'annual_flag',
    'annual_wage': 'annual_flag',
    'wage_comment_e': 'wage_comment',
}

# French language columns to drop
FRENCH_COLUMNS_TO_DROP = [
    'nom_re',
    'titre_cnp',
    'noc_title_f',
    'noc_title_fra',
    'noc_title_french',
    'data_source_f',
    'source_de_donnees',
    'wage_comment_f',
    'wage_comment_f1',
    'commentaire_salaire',
]

# Other columns to drop
OTHER_COLUMNS_TO_DROP = [
    'non_wageben_pct',
    'reference_period',
    'revision_date',
    'revision_date_date_revision',
    'reference_period_periode_de_reference',
    'revision_date_date_revision',
    'data_source_e',
    'data_source',
    'prov',
    'average_wage_salaire_moyen',
    'quartile1_wage_salaire_quartile1',
    'quartile3_wage_salaire_quartile3',
]

# Process each dataframe

standard_col_dfs = []

for df, year in dataframes:
    df = df.copy()
    
    # Add year as the first column
    df['year'] = year
    df = df[['year'] + [col for col in df.columns if col != 'year']]
    
    # Standardize all column names casing
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    # Drop French columns
    cols_to_drop = [col for col in FRENCH_COLUMNS_TO_DROP if col in df.columns]
    if cols_to_drop:
        df.drop(columns=cols_to_drop, inplace=True)
    
    # Drop other unwanted columns
    cols_to_drop = [col for col in OTHER_COLUMNS_TO_DROP if col in df.columns]
    if cols_to_drop:
        df.drop(columns=cols_to_drop, inplace=True)
    # in 2016, there is an empty 'Annual_Wage_Flag_Salaire_annuel' column
    if year == 2016:
        df.drop(columns='annual_wage_flag_salaire_annuel', inplace=True)
    
    # Rename the columns to adhere to the single naming standard
    for old_col, new_col in RENAME_MAP.items():
        if old_col in df.columns:
            df.rename(columns={old_col: new_col}, inplace=True)

    print(f"{year} - Final columns: {df.columns.tolist()}")
    standard_col_dfs.append(df)


# Combine all dataframes
wages_df = pd.concat(standard_col_dfs, ignore_index=True)


print("\nCOMBINED DATAFRAME INFO")
print("="*80)
print(f"Dimensions: {wages_df.shape}")
wages_df.head()

2012 - Final columns: ['year', 'noc_code', 'noc_title', 'er_code', 'er_name', 'low_wage', 'median_wage', 'high_wage', 'annual_flag', 'wage_comment']
2013 - Final columns: ['year', 'noc_code', 'noc_title', 'er_code', 'er_name', 'low_wage', 'median_wage', 'high_wage', 'annual_flag', 'wage_comment']
2014 - Final columns: ['year', 'noc_code', 'noc_title', 'er_code', 'er_name', 'low_wage', 'median_wage', 'high_wage', 'annual_flag', 'wage_comment']
2015 - Final columns: ['year', 'noc_code', 'noc_title', 'er_code', 'er_name', 'low_wage', 'median_wage', 'high_wage', 'annual_flag', 'wage_comment']
2016 - Final columns: ['year', 'noc_code', 'noc_title', 'er_code', 'er_name', 'low_wage', 'median_wage', 'high_wage', 'wage_comment', 'annual_flag']
2017 - Final columns: ['year', 'noc_code', 'noc_title', 'er_code', 'er_name', 'low_wage', 'median_wage', 'high_wage', 'annual_flag', 'wage_comment']
2018 - Final columns: ['year', 'noc_code', 'noc_title', 'er_code', 'er_name', 'low_wage', 'median_wage', '

,year,noc_code,noc_title,er_code,er_name,low_wage,median_wage,high_wage,annual_flag,wage_comment
0,2012,11,Legislators,ER00,National,8.17,28.00,49.45,0,NaN
1,2012,11,Legislators,ER10,Newfoundland and Labrador,17.95,40.06,64.97,0,NaN
2,2012,11,Legislators,ER1010,Avalon Peninsula,19.23,45.67,66.67,0,NaN
3,2012,11,Legislators,ER1020,South Coast--Burin Peninsula,NaN,NaN,NaN,0,"Due to data limitations, the wage for this occ..."
4,2012,11,Legislators,ER1030,West Coast--Northern Peninsula--Labrador,NaN,NaN,NaN,0,"Due to data limitations, the wage for this occ..."


We can see that the dataset standardization and merging were successful.

#### **Narrow the scope**

To align with the scope of our national-level AI impact analysis, we will focus on the country-level data.

In [97]:
# We'll focus on national-level data to match the AI adoption scope
# Canada = Economic Region - 'ER00'
wages_df = wages_df[wages_df['er_code'] == 'ER00'].copy()

print(wages_df.head().to_string())

# Check the Economic Region name consistency
print(f'\n {wages_df['er_name'].unique()}')

     year noc_code                                                                                        noc_title er_code   er_name  low_wage  median_wage  high_wage  annual_flag wage_comment
0    2012       11                                                                                      Legislators    ER00  National      8.17        28.00      49.45            0          NaN
86   2012       12                                                         Senior Government Managers and Officials    ER00  National     20.64        46.15      74.36            0          NaN
172  2012       13                          Senior Managers - Financial, Communications and Other Business Services    ER00  National     20.29        40.87      75.83            0          NaN
258  2012       14  Senior Managers - Health, Education, Social and Community Services and Membership Organizations    ER00  National     19.23        38.46      66.67            0          NaN
344  2012       15            

In [98]:
# Replace all instances of 'National' with 'Canada' in the 'er_name' column
wages_df['er_name'] = wages_df['er_name'].replace('National', 'Canada')

print('The wages_df dimensions:', wages_df.shape)
print('\n', wages_df.groupby('year').size())

The wages_df dimensions: (6612, 10)

 year
2012    520
2013    520
2014    520
2015    520
2016    500
2017    500
2018    500
2019    500
2020    500
2021    500
2022    500
2023    516
2024    516
dtype: int64


With the scope narrowed, the final DataFrame contains 500 to 520 corresponding occupation records for each year of the analysis.

#### **Check the annual wages / convert to hourly rates**

In [99]:
# Check the stats for the annual wages
annual_wages_df = wages_df[(wages_df['annual_flag'] == 1)]
annual_wages_df.shape

(172, 10)

In [100]:
# The number of occupations with an annual wage rate per year
print("Records with annual wage rate per year:")
print(annual_wages_df.groupby('year').size())

Records with annual wage rate per year:
year
2012     8
2013     9
2014    12
2015    13
2016    13
2017    13
2018    13
2019    13
2020    14
2021    14
2022    14
2023    18
2024    18
dtype: int64


In [101]:
# Check unique wage comments for the occupations with annual wage rate
print(annual_wages_df[(annual_wages_df['low_wage'].notna())]['wage_comment'].unique())

['In this occupation, a large number of people are self-employed. For this reason, wages for this occupation are presented at an annual rate to better represent earnings for this occupation.'
 nan
 'In this occupation, a large number of people are self-employed. For this reason, wages for this occupation are presented at an annual rate to better repre'
 'In this occupation, a large number of people are self-employed. For this reason, wages for this occupation are presented at an annual rate to better represent earnings for this occupation. \n'
 'Wages for this occupation are presented at an annual rate to better represent earnings for this occupation.']


Only **8 to 18 occupations per year** (**±3.6% of the total data**) are reported with an annual wage rate. Based on the wage comments, this rate is primarily applied to occupations characterized by high self-employment, where employees typically work irregular hours and an annual figure is used to best represent total earnings.

Given that over 96% of the data is already reported as an hourly wage rate, we decided to standardize the remaining small fraction of annual rates by converting them to hourly units using a standard assumption of **2,000 working hours (40 hours/week×50 weeks/year)**.

- Converting the few annual rates to hourly units is necessary to prevent the exclusion of these occupations and enable direct comparison with the majority of hourly-reported wages.

- For the ≈8 to 18 occupations reported annually due to high self-employment, the resulting hourly wage is an **imputed** (approximate/substitute) value. Although this value relies on the 2,000-hour assumption and may not perfectly reflect the actual irregular working hours, it represents the most practical method for maintaining a consistent hourly metric.

In [102]:
# Standardize all wages to be hourly. Annual wages are divided by 2000 (40h/wk * 50wk/yr).
wage_cols = ['low_wage', 'median_wage', 'high_wage']
for col in wage_cols:
    wages_df[col] = np.where(wages_df['annual_flag'] == 1, wages_df[col] / 2000, wages_df[col])

#### **Check how the NOC codes are represented**

In [103]:
# Convert to string to ensure consistent handling of NOC codes
wages_df['noc_code'] = wages_df['noc_code'].astype(str)

noc_codes = wages_df['noc_code'].unique()

print(f"Total Unique NOC Codes Found: {len(noc_codes)}")
print("-" * 30)

# Print all unique NOC codes
with np.printoptions(threshold=np.inf):
    print(noc_codes)

Total Unique NOC Codes Found: 1709
------------------------------
['11' '12' '13' '14' '15' '16' '111' '112' '113' '114' '121' '122' '123'
 '131' '132' '211' '212' '213' '311' '312' '313' '314' '411' '412' '413'
 '414' '511' '512' '513' '611' '621' '631' '632' '641' '642' '643' '651'
 '711' '712' '713' '721' '811' '911' '912' '1111' '1112' '1113' '1114'
 '1121' '1122' '1211' '1212' '1213' '1214' '1215' '1221' '1222' '1223'
 '1224' '1225' '1226' '1227' '1228' '1231' '1232' '1233' '1234' '1235'
 '1236' '1241' '1242' '1243' '1244' '1411' '1413' '1414' '1422' '1423'
 '1424' '1431' '1432' '1433' '1434' '1435' '1441' '1442' '1443' '1451'
 '1452' '1453' '1454' '1461' '1462' '1463' '1471' '1472' '1473' '1474'
 '1475' '1476' '2111' '2112' '2113' '2114' '2115' '2121' '2122' '2123'
 '2131' '2132' '2133' '2134' '2141' '2142' '2143' '2144' '2145' '2146'
 '2147' '2148' '2151' '2152' '2153' '2154' '2161' '2171' '2172' '2173'
 '2174' '2175' '2211' '2212' '2213' '2221' '2222' '2223' '2224' '2225'
 '223

For consistency and ease of later analysis, we will ensure the NOC codes contain only the numeric values (by stripping any textual prefixes).   
We can also see that some NOC codes are shorter than expected (< 4 digits), indicating that leading zeros were dropped.These zeros are essential for identifying the major occupational group (the first digit).

In [104]:
# Clean the NOC codes
wages_df['noc_code'] = wages_df['noc_code'].str.replace('NOC_', '', case=False, regex=False)

# If code length is 2 or 3 -> pad to 4 digits (e.g., "11" -> "0011") 
wages_df['noc_code'] = wages_df['noc_code'].apply(lambda x: x.zfill(4) if len(x) <= 3 else x)

# Check the NOC codes
noc_codes = wages_df['noc_code'].unique()
# Print all unique NOC codes
with np.printoptions(threshold=np.inf):
    print(noc_codes)

['0011' '0012' '0013' '0014' '0015' '0016' '0111' '0112' '0113' '0114'
 '0121' '0122' '0123' '0131' '0132' '0211' '0212' '0213' '0311' '0312'
 '0313' '0314' '0411' '0412' '0413' '0414' '0511' '0512' '0513' '0611'
 '0621' '0631' '0632' '0641' '0642' '0643' '0651' '0711' '0712' '0713'
 '0721' '0811' '0911' '0912' '1111' '1112' '1113' '1114' '1121' '1122'
 '1211' '1212' '1213' '1214' '1215' '1221' '1222' '1223' '1224' '1225'
 '1226' '1227' '1228' '1231' '1232' '1233' '1234' '1235' '1236' '1241'
 '1242' '1243' '1244' '1411' '1413' '1414' '1422' '1423' '1424' '1431'
 '1432' '1433' '1434' '1435' '1441' '1442' '1443' '1451' '1452' '1453'
 '1454' '1461' '1462' '1463' '1471' '1472' '1473' '1474' '1475' '1476'
 '2111' '2112' '2113' '2114' '2115' '2121' '2122' '2123' '2131' '2132'
 '2133' '2134' '2141' '2142' '2143' '2144' '2145' '2146' '2147' '2148'
 '2151' '2152' '2153' '2154' '2161' '2171' '2172' '2173' '2174' '2175'
 '2211' '2212' '2213' '2221' '2222' '2223' '2224' '2225' '2231' '2232'
 '2233

 The formatting of the NOC codes is now standardized.

#### **Missing values handling**

 Check the number of missing values per column

In [105]:
# Check the number of missing values per column
print(wages_df.isna().sum())

year               0
noc_code           0
noc_title          0
er_code            0
er_name            0
low_wage           5
median_wage        5
high_wage          5
annual_flag        0
wage_comment    6447
dtype: int64


In [106]:
# Define the wage columns to check for missing values
wage_cols = ['low_wage', 'median_wage', 'high_wage']

# Create a boolean mask: True if ANY of the wage columns is null (NaN)
nan_wage_mask = wages_df[wage_cols].isnull().any(axis=1)

# Filter the DataFrame using the mask and display the head of the resulting records
print("Records where at least one wage column is missing (NaN):")
print(wages_df[nan_wage_mask].head(20).to_string())


Records where at least one wage column is missing (NaN):
        year noc_code                                                                                        noc_title er_code er_name  low_wage  median_wage  high_wage  annual_flag                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 wage_comment
279156  2018     3111                                                                            Specialist physicians    ER00  Canada       NaN          NaN        NaN            1  Due to data limitatio


The rows with missing wage values are all subject to data suppression by the source. As confirmed by the comments, wages are omitted because these occupations have unique characteristics that make it difficult to accurately ascertain their wage information.

Since the actual wages are unknown and can't be guessed accurately and comprise only a small fraction of all data (only 5 out of over 6,600 records have missing wages), the best choice is to drop these records.

In [107]:
# Drop the records with the missing wage values
wages_df.dropna(subset=wage_cols, how='all', inplace=True)

#### **Sanity check: summary statistics**

In [108]:
# Define the wage columns
wage_cols = ['low_wage', 'median_wage', 'high_wage']

# Group the DataFrame by year and apply the .describe() method to get summary statistics
wage_summary = wages_df.groupby('year')[wage_cols].describe()

print("Summary Statistics by Year for All Wages (Low, Median, and High):")
print(wage_summary.to_string())

Summary Statistics by Year for All Wages (Low, Median, and High):
     low_wage                                                                          median_wage                                                                          high_wage                                                                         
        count       mean          std       min      25%     50%      75%          max       count        mean          std    min      25%     50%        75%          max     count        mean          std    min      25%     50%      75%            max
year                                                                                                                                                                                                                                                          
2012    520.0  14.429802     4.266255   7.50000  11.0000  14.000  17.0000     33.52000       520.0   24.113750     8.940115  10.00  17.5000  22.680  29.075000     82.275

The are abnormally high values in the 2013 summary statistics, so we need to locate the outliers responsible for the extreme right skew.

In [109]:
# Filter the DataFrame for the year 2013 AND a high_wage to find the outlier(s)
anomaly_records = wages_df[(wages_df['year'] == 2013) & (wages_df['high_wage'] > 500)] # Use a high threshold to capture the annual salary outliers

print("Record(s) causing the 2013 wage anomaly:")
# Display relevant columns to confirm the error and identify the NOC code
print(anomaly_records[['year', 'noc_code', 'noc_title', 'low_wage', 'median_wage', 'high_wage', 'annual_flag']].to_string())

Record(s) causing the 2013 wage anomaly:
       year noc_code                                                         noc_title  low_wage  median_wage  high_wage  annual_flag
59254  2013     3123  Other Professional Occupations in Health Diagnosing and Treating   21312.9      41658.0   107062.0            0
60802  2013     3221                                                        Denturists   22765.6      53279.0   101544.0            0


A data error in 2013 was caused by two specific records. Since the wage values exceed $20,000 (for the low wage) / $100,000 (for high wage), we believe that they are most likely unconverted annual salaries that were incorrectly marked as hourly in the source file (annual_flag=0). To correct this error, we will manually force the conversion of these two specific records to hourly rates by dividing their values by the 2,000-hour standard.

In [110]:
# Fix the abnormal records
wages_df.loc[
    (wages_df['year'] == 2013) & (wages_df['noc_code'].isin(['3123', '3221'])), 
    wage_cols
] = wages_df.loc[
    (wages_df['year'] == 2013) & (wages_df['noc_code'].isin(['3123', '3221'])), 
    wage_cols
] / 2000

# Group the DataFrame by year and apply the .describe() method to get summary statistics
wage_summary = wages_df.groupby('year')[wage_cols].describe()

print("Summary Statistics by Year for All Wages (Low, Median, and High):")
print(wage_summary.to_string())

Summary Statistics by Year for All Wages (Low, Median, and High):
     low_wage                                                                     median_wage                                                                     high_wage                                                                   
        count       mean       std       min      25%     50%      75%        max       count       mean        std    min      25%     50%        75%        max     count       mean        std    min      25%     50%      75%         max
year                                                                                                                                                                                                                                          
2012    520.0  14.429802  4.266255   7.50000  11.0000  14.000  17.0000   33.52000       520.0  24.113750   8.940115  10.00  17.5000  22.680  29.075000   82.27550     520.0  37.487837  16.042033  12.00  27.0350  34.985

After checking the summary statistics, we can see that the wages are logicaly consistent across all years.

#### **Check the remaining wage comments**

In [111]:
# Check the unique wage comments still present after cleaning
print(wages_df['wage_comment'].unique())

[nan
 'In this occupation, a large number of people are self-employed. For this reason, wages for this occupation are presented at an annual rate to better represent earnings for this occupation.'
 'In this occupation, a large number of people are self-employed. For this reason, wages for this occupation are presented at an annual rate to better repre'
 'In this occupation, a large number of people are self-employed. For this reason, wages for this occupation are presented at an annual rate to better represent earnings for this occupation. \n'
 'Wages for this occupation are presented at an annual rate to better represent earnings for this occupation.']


Since all remaining information in the `wage_comment` column relates solely to the representation of annual wage rates, and we’ve already completed all necessary adjustments and conversions for annual salaries, we can now safely remove both the `wage_comment` and `annual_flag` columns from the DataFrame.

In [112]:
# Drop the wage_comment and annual_flag columns as they are no longer needed
wages_df.drop(columns=['wage_comment', 'annual_flag'], inplace=True)

wages_df.head()

,year,noc_code,noc_title,er_code,er_name,low_wage,median_wage,high_wage
0,2012,0011,Legislators,ER00,Canada,8.17,28.00,49.45
86,2012,0012,Senior Government Managers and Officials,ER00,Canada,20.64,46.15,74.36
172,2012,0013,"Senior Managers - Financial, Communications an...",ER00,Canada,20.29,40.87,75.83
258,2012,0014,"Senior Managers - Health, Education, Social an...",ER00,Canada,19.23,38.46,66.67
344,2012,0015,"Senior Managers - Trade, Broadcasting and Othe...",ER00,Canada,18.90,40.87,72.12


# AI Adoption ([Statistics Canada - Use of Use of advanced or emerging technologies, by industry and enterprise size](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=2710036701))

In [113]:
advanced_tech_df = pd.read_csv("../datasets/advanced_tech/Use-of-advanced-tech_Open-Canada.csv")
advanced_tech_df.head()

,REF_DATE,GEO,DGUID,North American Industry Classification System (NAICS),Enterprise size,Advanced or emerging technologies,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2017,Canada,2021A000011124,"Total, all surveyed industries","Total, all enterprise sizes",Any advanced or emerging technology use,Percent,239,units,0,v1047697746,1.1.1.17,46.1,A,NaN,NaN,1
1,2017,Canada,2021A000011124,"Total, all surveyed industries","Total, all enterprise sizes",Advanced technology use,Percent,239,units,0,v1047697730,1.1.1.1,42.8,A,NaN,t,1
2,2017,Canada,2021A000011124,"Total, all surveyed industries","Total, all enterprise sizes","Material handling, supply chain or logistics t...",Percent,239,units,0,v1047697731,1.1.1.2,12.7,A,NaN,NaN,1
3,2017,Canada,2021A000011124,"Total, all surveyed industries","Total, all enterprise sizes",Design or information control technologies,Percent,239,units,0,v1047697732,1.1.1.3,17.7,A,NaN,NaN,1
4,2017,Canada,2021A000011124,"Total, all surveyed industries","Total, all enterprise sizes",Processing or fabrication technologies,Percent,239,units,0,v1047697733,1.1.1.4,12.2,A,NaN,NaN,1


## **Transformation**

In [114]:
# Filter for metrics for all enterprise sizes, as enterprise size is not relevant for our analysis
# Focus on 'Artificial intelligence (AI)' use
ai_adoption_df = advanced_tech_df[
    (advanced_tech_df['Advanced or emerging technologies'] == 'Artificial intelligence (AI)') &
    (advanced_tech_df['Enterprise size'] == 'Total, all enterprise sizes')
].copy()

ai_adoption_df.head()

,REF_DATE,GEO,DGUID,North American Industry Classification System (NAICS),Enterprise size,Advanced or emerging technologies,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
12,2017,Canada,2021A000011124,"Total, all surveyed industries","Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047697742,1.1.1.13,4.0,A,NaN,NaN,1
80,2017,Canada,2021A000011124,"Agriculture, forestry, fishing and hunting [11]","Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047697810,1.2.1.13,1.8,A,NaN,NaN,1
148,2017,Canada,2021A000011124,"Mining, quarrying, and oil and gas extraction ...","Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047697878,1.3.1.13,3.7,A,NaN,NaN,1
216,2017,Canada,2021A000011124,Oil and gas extraction [211],"Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047697946,1.4.1.13,2.1,A,NaN,NaN,1
284,2017,Canada,2021A000011124,Mining and quarrying (except oil and gas) [212],"Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047698014,1.5.1.13,3.3,A,NaN,NaN,1


#### Major **NAICS** industry sectors

| Code | Sector |
| :--- | :--- |
| 11 | Agriculture, forestry, fishing and hunting |
| 21 | Mining, quarrying, and oil and gas extraction |
| 22 | Utilities |
| 23 | Construction |
| 31-33 | Manufacturing |
| 41 | Wholesale trade |
| 44-45 | Retail trade |
| 48-49 | Transportation and warehousing |
| 51 | Information and cultural industries |
| 52 | Finance and insurance |
| 53 | Real estate and rental and leasing |
| 54 | Professional, scientific and technical services |
| 55 | Management of companies and enterprises |
| 56 | Administrative and support, waste management and remediation services |
| 61 | Educational services |
| 62 | Health care and social assistance |
| 71 | Arts, entertainment and recreation |
| 72 | Accommodation and food services |
| 81 | Other services (except public administration) |
| 91 | Public administration |

To limit the scope of the analysis, we decided to only use the major (2-digit) NAISC industry sectors.  
Instead of ±100 niche industry categories, we will be comparing ±20 macro-sectors, this will help us reduce analytical noise and ensure that our conclusions represent broad trends across the Canadian economy.

In [115]:
# Use regex to extract the 2-digit NAISC sector code contained within [ ] and save it in a separate column
ai_adoption_df['naics_sector'] = ai_adoption_df['North American Industry Classification System (NAICS)'].str.extract(r'\[(.*?)\]')

print('The NAISC sectors present in the dataset:')
print(ai_adoption_df['naics_sector'].unique())

The NAISC sectors present in the dataset:
[nan '11' '21' '211' '212' '213' '22' '23' '31-33' '311' '312' '321' '322'
 '323' '324' '325' '3254' '326' '3261' '326193' '3262' '327' '331' '332'
 '333' '334' '3344' '3345' '335' '336' '3363' '3364' '337' '339' '3391'
 '3399' '41' '411' '412' '413' '414' '4145' '415' '416' '417' '4173' '418'
 '419' '44-45' '48-49' '481' '484' '485' '486' '51' '511' '5111' '5112'
 '512' '515' '517' '518' '519' '523' '5231' '524' '53' '54' '5411' '5412'
 '5413' '5415' '5416' '5417' '5418' '55' '56' '3117']


In [116]:
# These are the major (2-digit) sectors that are present in the dataset
target_major_codes = [ '11', '21', '22', '23', '31-33', '41', '44-45', '48-49', '51', '52', '53', '54', '55', '56' ]

# Create filtering conditions
total_industry_filter = ai_adoption_df['North American Industry Classification System (NAICS)'] == 'Total, all surveyed industries'
major_sector_filter = ai_adoption_df['naics_sector'].isin(target_major_codes)

# Apply filter to keep only major sectors OR the overall total row
ai_adoption_df = ai_adoption_df[major_sector_filter | total_industry_filter]

# Replace missing naics_sector values for the 'Total, all surveyed industries' records with 'TOTAL'
ai_adoption_df['naics_sector'] = ai_adoption_df['naics_sector'].fillna('TOTAL')

print(ai_adoption_df.head().to_string())
print(ai_adoption_df['naics_sector'].unique())

     REF_DATE     GEO           DGUID North American Industry Classification System (NAICS)              Enterprise size Advanced or emerging technologies      UOM  UOM_ID SCALAR_FACTOR  SCALAR_ID       VECTOR COORDINATE  VALUE STATUS  SYMBOL TERMINATED  DECIMALS naics_sector
12       2017  Canada  2021A000011124                        Total, all surveyed industries  Total, all enterprise sizes      Artificial intelligence (AI)  Percent     239         units          0  v1047697742   1.1.1.13    4.0      A     NaN        NaN         1        TOTAL
80       2017  Canada  2021A000011124       Agriculture, forestry, fishing and hunting [11]  Total, all enterprise sizes      Artificial intelligence (AI)  Percent     239         units          0  v1047697810   1.2.1.13    1.8      A     NaN        NaN         1           11
148      2017  Canada  2021A000011124    Mining, quarrying, and oil and gas extraction [21]  Total, all enterprise sizes      Artificial intelligence (AI)  Percent     2

In [117]:
# Rename columns for more clarity
ai_adoption_df.rename(columns={
    'REF_DATE': 'year',
    'North American Industry Classification System (NAICS)': 'naics_name',
    'VALUE': 'ai_adoption_rate',
}, inplace=True)

ai_adoption_df.head()

,year,GEO,DGUID,naics_name,Enterprise size,Advanced or emerging technologies,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,ai_adoption_rate,STATUS,SYMBOL,TERMINATED,DECIMALS,naics_sector
12,2017,Canada,2021A000011124,"Total, all surveyed industries","Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047697742,1.1.1.13,4.0,A,NaN,NaN,1,TOTAL
80,2017,Canada,2021A000011124,"Agriculture, forestry, fishing and hunting [11]","Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047697810,1.2.1.13,1.8,A,NaN,NaN,1,11
148,2017,Canada,2021A000011124,"Mining, quarrying, and oil and gas extraction ...","Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047697878,1.3.1.13,3.7,A,NaN,NaN,1,21
420,2017,Canada,2021A000011124,Utilities [22],"Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047698150,1.7.1.13,3.4,A,NaN,NaN,1,22
488,2017,Canada,2021A000011124,Construction [23],"Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047698218,1.8.1.13,0.8,A,NaN,NaN,1,23


## **Cleaning**

The `Status` column indicates data quality. To ensure the accuracy of our analysis, we decided to investigate the data quality.

#### Symbol Legend (from the dataset's metadata)
| Description | Symbol |
--------------|--------|
| "data quality: excellent" | "A" |
| "data quality: very good" | "B" |
| "data quality: good" | "C" |
| "data quality: acceptable" | "D" |
| "use with caution" | "E" |
| "too unreliable to be published" | "F" |

In [118]:
# Check the data quality of the present records
print(ai_adoption_df['STATUS'].unique())
print(ai_adoption_df['STATUS'].value_counts())

# Inspect the records with quality below 'acceptable'
inspect_df = ai_adoption_df[(ai_adoption_df['STATUS'] == 'E') | (ai_adoption_df['STATUS'] == 'F')]

inspect_df.head(50)

['A' 'B' 'E' 'C' 'F']
STATUS
A    143
B     69
E     20
C      4
F      2
Name: count, dtype: int64


,year,GEO,DGUID,naics_name,Enterprise size,Advanced or emerging technologies,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,ai_adoption_rate,STATUS,SYMBOL,TERMINATED,DECIMALS,naics_sector
6574,2017,Atlantic Region,2021A00011,Utilities [22],"Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047704304,2.7.1.13,0.0,E,NaN,NaN,1,22
7883,2017,Atlantic Region,2021A00011,Real estate and rental and leasing [53],"Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047705613,2.84.1.13,0.0,E,NaN,NaN,1,53
8053,2017,Atlantic Region,2021A00011,Management of companies and enterprises [55],"Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047705783,2.94.1.13,0.0,E,NaN,NaN,1,55
8070,2017,Atlantic Region,2021A00011,"Administrative and support, waste management a...","Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047705800,2.95.1.13,0.0,E,NaN,NaN,1,56
8121,2017,Quebec,2021A000224,"Mining, quarrying, and oil and gas extraction ...","Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047705851,3.3.1.13,0.0,E,NaN,NaN,1,21
20190,2019,Atlantic Region,2021A00011,"Mining, quarrying, and oil and gas extraction ...","Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047704236,2.3.1.13,0.0,E,NaN,NaN,1,21
20285,2019,Atlantic Region,2021A00011,Construction [23],"Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047704321,2.8.1.13,0.0,E,NaN,NaN,1,23
21729,2019,Atlantic Region,2021A00011,Real estate and rental and leasing [53],"Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047705613,2.84.1.13,0.0,E,NaN,NaN,1,53
21919,2019,Atlantic Region,2021A00011,Management of companies and enterprises [55],"Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047705783,2.94.1.13,0.0,E,NaN,NaN,1,55
23781,2019,Ontario,2021A000235,"Agriculture, forestry, fishing and hunting [11]","Total, all enterprise sizes",Artificial intelligence (AI),Percent,239,units,0,v1047707449,4.2.1.13,0.0,E,NaN,NaN,1,11


Most "use with caution" entries are tied to Atlantic Region.  
The dataset doesn't have the full province-by-province breakdown (that's there regional aggregates like 'Atlantic Region' come from), so to optimize for data quality and to better align with the scope of our datasets, we decided to only keep the values for Canada.

In [119]:
# Filter to only include the national level stats
ai_adoption_df = ai_adoption_df[(ai_adoption_df['GEO'] == 'Canada')]

print(ai_adoption_df['STATUS'].unique())
print(ai_adoption_df['STATUS'].value_counts())

#Check Na values
print(ai_adoption_df.isna().sum())


['A' 'B']
STATUS
A    38
B     4
Name: count, dtype: int64
year                                  0
GEO                                   0
DGUID                                 0
naics_name                            0
Enterprise size                       0
Advanced or emerging technologies     0
UOM                                   0
UOM_ID                                0
SCALAR_FACTOR                         0
SCALAR_ID                             0
VECTOR                                0
COORDINATE                            0
ai_adoption_rate                      0
STATUS                                0
SYMBOL                               42
TERMINATED                           42
DECIMALS                              0
naics_sector                          0
dtype: int64


After narrowing the scope, the `NaN` values and the values with lower data quality are absent.

Now we will remove the columns that are not needed for our analysis to make the dataset more readable and easier to work with.

In [120]:
ai_adoption_df = ai_adoption_df.drop(columns= [
    'Enterprise size', 'Advanced or emerging technologies', 'UOM', #all the same known values
    'DGUID','UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS', # not needed 
    ]) 

#### Finally, check the AI adoption rate summary statistics.

In [121]:
# Check the summary statistics
ai_adoption_df.groupby('year')['ai_adoption_rate'].describe()

,count,mean,std,min,25%,50%,75%,max
year,,,,,,,,
2017,14.0,4.392857,4.385372,0.8,2.100,3.10,3.925,16.8
2019,14.0,6.228571,5.947130,0.9,3.150,4.65,6.050,20.9
2022,14.0,7.814286,5.221090,2.2,4.825,5.80,7.600,18.0


AI adoption rates across Canadian NAICS sectors show a steady upward shift from 2017 to 2022, with means and medians rising each year and similar levels of variability, indicating broad but uneven adoption.  
The range widens at the top end (right-skewed), but overall the distribution appears to be consistent, with no clear signs of abnormal values.

In [122]:
# Select final columns to keep
final_columns = ['year', 'GEO', 'naics_name', 'naics_sector', 'ai_adoption_rate']
ai_adoption_df = ai_adoption_df[final_columns]
ai_adoption_df.head()

,year,GEO,naics_name,naics_sector,ai_adoption_rate
12,2017,Canada,"Total, all surveyed industries",TOTAL,4.0
80,2017,Canada,"Agriculture, forestry, fishing and hunting [11]",11,1.8
148,2017,Canada,"Mining, quarrying, and oil and gas extraction ...",21,3.7
420,2017,Canada,Utilities [22],22,3.4
488,2017,Canada,Construction [23],23,0.8


# ---------------------------------------------------------------------------------------------

## **Exports**

In [123]:
wages_df.to_csv("../datasets_cleaned/wages.csv", index=False)
ai_adoption_df.to_csv("../datasets_cleaned/ai_adoption.csv", index=False)